##IMPORT

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
from google.colab import drive

# 1. Mount Google Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# 2. Configuration - UPDATE THESE PATHS

In [ ]:
ROOT_PATH = '/content/drive/MyDrive/Distance_classification'  # Folder containing 10, 20, 30, 40, 50
REFERENCE_IMAGE_PATH = '/content/drive/MyDrive/Distance_classification/reference.jpg' # Your reference image
IMG_SIZE = 224
ROI = (50, 200, 50, 200) # y1, y2, x1, x2 (Adjust based on your speckle location)

# 3. Dataset Logic with ZNCC & ROI

In [ ]:
class SpeckleDistanceDataset(Dataset):
    def __init__(self, root_dir, ref_path, img_size=IMG_SIZE, roi_coords=ROI):
        self.root_dir = root_dir
        self.roi = roi_coords
        self.img_size = img_size
        self.classes = ['10', '20', '30', '40', '50']
        self.data = []

        ref_img = Image.open(ref_path).convert('L')
        ref_tensor = transforms.ToTensor()(ref_img)
        self.ref_roi = self._extract_roi(ref_tensor)

        for idx, cls in enumerate(self.classes):
            cls_folder = os.path.join(root_dir, cls)
            if not os.path.exists(cls_folder): continue
            for img_name in os.listdir(cls_folder):
                self.data.append((os.path.join(cls_folder, img_name), idx))

    def _extract_roi(self, img):
        y1, y2, x1, x2 = self.roi
        return img[:, y1:y2, x1:x2]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = Image.open(img_path).convert('L')
        img_tensor = transforms.ToTensor()(img)

        img_roi = self._extract_roi(img_tensor)

        # Zero-mean Normalization (ZNCC Pre-step)
        img_roi = (img_roi - img_roi.mean()) / (img_roi.std() + 1e-8)
        ref_roi_norm = (self.ref_roi - self.ref_roi.mean()) / (self.ref_roi.std() + 1e-8)

        # Stack as 2-channel input
        combined = torch.cat([img_roi, ref_roi_norm], dim=0)
        combined = transforms.Resize((self.img_size, self.img_size))(combined)

        return combined, label

# 4. ResNet-9 Architecture

In [ ]:
def conv_block(in_f, out_f, pool=False):
    layers = [nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
              nn.BatchNorm2d(out_f),
              nn.ReLU(inplace=True)]
    if pool: layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

class ResNet9(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv1 = conv_block(in_channels, 64)
        self.conv2 = conv_block(64, 128, pool=True)
        self.res1 = nn.Sequential(conv_block(128, 128), conv_block(128, 128))
        self.conv3 = conv_block(128, 256, pool=True)
        self.conv4 = conv_block(256, 512, pool=True)
        self.res2 = nn.Sequential(conv_block(512, 512), conv_block(512, 512))
        self.classifier = nn.Sequential(nn.AdaptiveMaxPool2d(1), nn.Flatten(),
                                        nn.Dropout(0.2), nn.Linear(512, num_classes))

    def forward(self, xb):
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out
        return self.classifier(out)

# 5. Training and Efficiency Loop

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
full_dataset = SpeckleDistanceDataset(ROOT_PATH, REFERENCE_IMAGE_PATH)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=4, shuffle=True)
test_loader = DataLoader(test_set, batch_size=4)

model = ResNet9(in_channels=2, num_classes=5).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, pred = torch.max(model(imgs), 1)
            correct += (pred == labels).sum().item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.3f} | Accuracy: {100*correct/test_size:.1f}%")

Epoch 1 | Loss: 4.115 | Accuracy: 33.3%
Epoch 2 | Loss: 0.758 | Accuracy: 0.0%
Epoch 3 | Loss: 0.147 | Accuracy: 0.0%
Epoch 4 | Loss: 0.085 | Accuracy: 0.0%
Epoch 5 | Loss: 0.088 | Accuracy: 0.0%
Epoch 6 | Loss: 0.029 | Accuracy: 0.0%
Epoch 7 | Loss: 0.021 | Accuracy: 0.0%
Epoch 8 | Loss: 0.007 | Accuracy: 0.0%
Epoch 9 | Loss: 0.003 | Accuracy: 0.0%
Epoch 10 | Loss: 0.004 | Accuracy: 0.0%
